# Fixed-camera annotation workset — Colab T4

下载两段 CC BY 4.0 固定机位视频，校验 SHA256，生成不修改原片的工作副本，并导出可导入 CVAT 的 YOLO + DeepSORT 预标注。预标注只作为人工复核起点，不能直接作为论文真值。

In [ ]:
# 1. 检查 GPU
import subprocess, sys
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), '请在 代码执行程序 → 更改运行时类型 中选择 T4 GPU'
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. 克隆或更新项目，并安装 Colab 依赖
from pathlib import Path
import os, subprocess

REPO_URL = 'https://github.com/mengdie305-hue/traffic-violation-detection.git'
PROJECT_DIR = Path('/content/traffic-violation-detection')
if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
# 3. 准备 YOLOv8n 权重；GitHub 波动时自动切换到 Ultralytics 的 Hugging Face 镜像
from pathlib import Path
import shutil, urllib.request
from ultralytics import YOLO

model_dir = Path('data/models')
model_dir.mkdir(parents=True, exist_ok=True)
target_model = model_dir / 'yolov8n.pt'
if not target_model.exists():
    try:
        YOLO('yolov8n.pt')
        shutil.copy2('yolov8n.pt', target_model)
    except Exception:
        mirror = 'https://huggingface.co/Ultralytics/YOLOv8/resolve/main/yolov8n.pt'
        urllib.request.urlretrieve(mirror, target_model)
YOLO(str(target_model))
print('Model:', target_model.resolve(), target_model.stat().st_size, 'bytes')

In [ ]:
# 4. 下载、校验并生成两个工作副本
from pathlib import Path
import hashlib, subprocess, sys, urllib.request

raw_dir = Path('data/raw/candidates/zenodo_14041920')
raw_dir.mkdir(parents=True, exist_ok=True)
items = [
    {'id': 'zenodo_night_001', 'url': 'https://zenodo.org/api/records/14041920/files/2024-11-04%2012-00-31.mp4/content', 'raw': raw_dir / '2024-11-04_12-00-31.mp4', 'sha256': '1205d0080225070edebc7c04ac37542bfe6922f540ec259118581978eff411ce', 'start': '3', 'end': '103.5', 'top': '32', 'bottom': '40'},
    {'id': 'zenodo_snow_001', 'url': 'https://zenodo.org/api/records/14041920/files/teste2_6.mp4/content', 'raw': raw_dir / 'teste2_6.mp4', 'sha256': 'ac131a355aeeb2493c170d1497ab172f0548786d1cda08e05236f4a5fe4fae82', 'start': '4', 'end': '95', 'top': '24', 'bottom': '0'},
]
for item in items:
    if not item['raw'].exists():
        print('Downloading', item['id'])
        urllib.request.urlretrieve(item['url'], item['raw'])
    digest = hashlib.sha256(item['raw'].read_bytes()).hexdigest()
    assert digest == item['sha256'], (item['id'], digest)
    item['work'] = Path('data/processed/annotation_workset') / f"{item['id']}.mp4"
    subprocess.run([sys.executable, 'scripts/prepare_annotation_video.py', '--input', str(item['raw']), '--output', str(item['work']), '--start', item['start'], '--end', item['end'], '--crop-top', item['top'], '--crop-bottom', item['bottom']], check=True)

In [ ]:
# 5. 使用 T4 生成 CVAT 1.1 轨迹预标注
from pathlib import Path
import subprocess, sys
for item in items:
    item['xml'] = Path('data/processed/preannotations') / f"{item['id']}.xml"
    subprocess.run([sys.executable, 'scripts/preannotate_cvat.py', '--video', str(item['work']), '--output', str(item['xml']), '--frame-step', '6', '--device', 'cuda'], check=True)

In [ ]:
# 6. 验证 XML、显示统计并下载预标注包
from pathlib import Path
from xml.etree import ElementTree as ET
from google.colab import files
import shutil

for item in items:
    root = ET.parse(item['xml']).getroot()
    tracks = len(root.findall('track'))
    boxes = len(root.findall('./track/box'))
    assert tracks > 0 and boxes > 0
    print(item['xml'].name, 'tracks=', tracks, 'keyframes=', boxes)
bundle = shutil.make_archive('/content/traffic_preannotations', 'zip', root_dir='data/processed', base_dir='preannotations')
print('Bundle:', bundle)
files.download(bundle)